<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_11_practicum_search/note_lesson_11_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 11 — Практикум П2. Пошук: зміна диспетчера таксі

У практикумі П1 диспетчерська таксі навчилася **рахувати ціну** рішення. Сьогодні в неї нове питання: **чи треба перебирати все?**

За зміну диспетчеру ставлять чотири питання. Кожне можна розв'язати перебором, але в кожному про дані щось відомо — і саме це робить розв'язок швидким:

| Питання | Що відомо про дані | Стратегія |
|---|---|---|
| чи їздив клієнт сьогодні? | нічого, журнал у порядку дзвінків | лінійний пошук |
| чи була поїздка о 18:40? | час відсортований | бінарний пошук |
| які дві поїздки дають рівно суму ваучера? | суми відсортовані | два вказівники |
| коли три найзавантаженіші години поспіль? | години йдуть підряд | ковзне вікно |

Наприкінці — автодоповнення адреси в застосунку таксі і дві задачі для самостійної роботи.

Як працювати: **передбач** відповідь, **запусти**, звір із передбаченням. Там, де написано `# YOUR CODE HERE`, допиши код — перевірки `assert` нижче скажуть, чи все правильно.

Теорія — у книзі: [Урок 11. Практикум П2. Пошук](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_11/).

## 🔁 Пригадай (без підглядання)

1. Скільки кроків робить функція, що перевіряє кожен елемент списку з n елементів, у найгіршому випадку?
2. Дослід подвоєння (урок 8) дав відношення ×2. Який це клас складності?
3. Що дасть `[10, 20, 30, 40][-1]` і `[10, 20, 30, 40][1:3]`?

<details>
<summary>Відповіді</summary>

1. n кроків: елемент останній або його немає — `O(n)`.
2. `O(n)`. Сьогодні побачимо клас, де подвоєння даних додає лише **один** крок.
3. `40` і `[20, 30]`.

</details>

## 1. Чи їздив клієнт сьогодні? — лінійний пошук

Журнал поїздок записаний у порядку дзвінків, імена йдуть як завгодно. Скористатися нічим: перевіряємо кожен запис. Функція повертає пару `(індекс, кроки)`; `-1` — не знайдено.

In [ ]:
def linear_search(items, target):
    steps = 0
    for i in range(len(items)):
        steps += 1
        if items[i] == target:
            return i, steps
    return -1, steps


clients = ["Марта", "Олег", "Ірина", "Тарас", "Олена", "Богдан"]
print(linear_search(clients, "Тарас"))
print(linear_search(clients, "Софія"))

assert linear_search(clients, "Тарас") == (3, 4)
assert linear_search(clients, "Софія") == (-1, 6)   # немає — перевірили всіх
assert linear_search([], "Тарас") == (-1, 0)

## 2. Чи була поїздка о 18:40? — бінарний пошук

Система записує час початку поїздок у хвилинах від опівночі (18:40 → `18 * 60 + 40 = 1120`). Записи додаються по порядку, тож список **відсортований**.

Дивимось на середній запис: якщо там 15:30, а шукаємо 18:40, уся ліва половина ще раніша — її можна відкинути. Одне порівняння — мінус половина даних.

**Передбач:** скільки кроків знадобиться на 1000 записах? На 2000? На мільйоні?

In [ ]:
def binary_search(items, target):
    low, high = 0, len(items) - 1
    steps = 0
    while low <= high:
        mid = (low + high) // 2
        steps += 1
        if items[mid] == target:
            return mid, steps
        if items[mid] < target:
            low = mid + 1
        else:
            high = mid - 1
    return -1, steps


starts = [425, 510, 612, 700, 845, 930, 1035, 1120, 1210, 1290, 1375]
print(binary_search(starts, 1120))
print(binary_search(starts, 1000))

assert binary_search(starts, 1120) == (7, 4)
assert binary_search(starts, 1000)[0] == -1

Як шукалося 1120 — пройди таблицю пальцем по списку `starts`:

| Крок | `low` … `high` | `mid` → значення | Рішення |
|---|---|---|---|
| 1 | 0 … 10 | 5 → 930 | менше → `low = 6` |
| 2 | 6 … 10 | 8 → 1210 | більше → `high = 7` |
| 3 | 6 … 7 | 6 → 1035 | менше → `low = 7` |
| 4 | 7 … 7 | 7 → 1120 | знайдено |

Тепер дослід подвоєння. Шукаємо значення, якого немає (найгірший випадок):

In [ ]:
for n in [1000, 2000, 4000, 1_000_000]:
    times = list(range(0, 2 * n, 2))
    _, steps = binary_search(times, -1)
    print(f"{n:>9} записів → {steps} кроків")

Подвоїли дані — **+1 крок**. Мільйон записів — 19 кроків (лінійному пошуку — мільйон). Кроків стільки, скільки разів n ділиться навпіл до одиниці: `log₂ n`. Це клас `O(log n)`.

⚠️ **Лише для відсортованих даних.** На невідсортованому списку пошук не падає — він мовчки бреше:

In [ ]:
print(binary_search([1120, 425, 930, 612], 1120))   # 1120 є в списку, але пошук її не знайде

### Готовий бінарний пошук: `bisect`

`bisect_left(список, x)` повертає позицію, куди можна вставити `x`, не зламавши порядку. Звідси відповідь і на «чи є», і на «скільки до».

**Завдання.** Порахуй, скільки поїздок почалося **з 18:00** (включно).

In [ ]:
from bisect import bisect_left

evening = 18 * 60

# YOUR CODE HERE
# BEGIN SOLUTION
index = bisect_left(starts, evening)
after_six = len(starts) - index
# END SOLUTION

print("поїздок з 18:00:", after_six)
assert after_six == 4


def contains(items, target):
    i = bisect_left(items, target)
    return i < len(items) and items[i] == target


assert contains(starts, 1120) and not contains(starts, 1000) and not contains(starts, 2000)
print("OK")

## 3. Ваучер на 500 грн — два вказівники

Пасажир отримав ваучер на **рівно 500 грн** і може покрити ним дві поїздки. Суми поїздок за тиждень **відсортовані**. Які дві поїздки дають рівно 500?

Перебір усіх пар — `O(n²)`. Два вказівники — з обох кінців назустріч:
- сума **замала** → лівий вказівник праворуч (до дорожчої поїздки);
- сума **завелика** → правий ліворуч (до дешевшої);
- рівно 500 → знайшли.

**Завдання.** Допиши тіло циклу. Функція повертає `(пара, кроки)` або `(None, кроки)`.

In [ ]:
fares = [120, 150, 180, 230, 270, 320, 410]


def pair_with_sum(items, target):
    left, right = 0, len(items) - 1
    steps = 0
    while left < right:
        steps += 1
        # YOUR CODE HERE
        # BEGIN SOLUTION
        total = items[left] + items[right]
        if total == target:
            return (items[left], items[right]), steps
        if total < target:
            left += 1
        else:
            right -= 1
        # END SOLUTION
    return None, steps


def pair_with_sum_brute(items, target):
    steps = 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            steps += 1
            if items[i] + items[j] == target:
                return (items[i], items[j]), steps
    return None, steps


print("два вказівники:", pair_with_sum(fares, 500))
print("перебір пар:   ", pair_with_sum_brute(fares, 500))
print("пари немає:    ", pair_with_sum(fares, 10_000), "проти", pair_with_sum_brute(fares, 10_000))

assert pair_with_sum(fares, 500) == ((180, 320), 4)
assert pair_with_sum(fares, 10_000) == (None, 6)
assert pair_with_sum([200, 300], 500) == ((200, 300), 1)
assert pair_with_sum([], 500) == (None, 0)
print("OK")

Чому можна відкинути 410 на першому кроці? `120 + 410 = 530` — забагато навіть з **найдешевшою** поїздкою, а з дорожчими буде ще більше. Кожен крок відкидає одну поїздку назавжди: щонайбільше n − 1 кроків, `O(n)`.

Без сортування ця логіка не працює. Ця сама задача повернеться в Практикумі 3 з невідсортованими даними — і там на допомогу прийде множина.

## 4. Пікові години — ковзне вікно

Кількість поїздок по годинах доби (індекс — година). Коли почати пікову зміну водіїв: які **три години поспіль** найзавантаженіші?

Наївний спосіб рахує суму кожного вікна заново — `O(n·k)`. Але сусідні вікна відрізняються двома годинами: одна випадає зліва, одна додається справа.

**Завдання.** Допиши `busiest_window`: оновлюй суму вікна, а не рахуй її заново. Результат має збігатися з наївною версією для будь-якого k.

In [ ]:
per_hour = [3, 1, 0, 0, 1, 4, 12, 30, 41, 25, 18, 20,
            22, 19, 17, 21, 33, 45, 52, 38, 24, 15, 9, 6]


def busiest_window_naive(counts, k):
    best_start, best_sum = 0, sum(counts[0:k])
    for start in range(1, len(counts) - k + 1):
        window_sum = sum(counts[start:start + k])
        if window_sum > best_sum:
            best_start, best_sum = start, window_sum
    return best_start, best_sum


def busiest_window(counts, k):
    window_sum = sum(counts[0:k])
    best_start, best_sum = 0, window_sum
    for end in range(k, len(counts)):
        # YOUR CODE HERE
        # BEGIN SOLUTION
        window_sum += counts[end] - counts[end - k]
        if window_sum > best_sum:
            best_start, best_sum = end - k + 1, window_sum
        # END SOLUTION
    return best_start, best_sum


print(busiest_window(per_hour, 3))
assert busiest_window(per_hour, 3) == (17, 135)      # 17:00–20:00, 135 поїздок
for k in range(1, 25):
    assert busiest_window(per_hour, k) == busiest_window_naive(per_hour, k), k
print("OK — збігається з наївною версією для k від 1 до 24")

## 5. Автодоповнення адреси

Пасажир вводить вулицю, і після кожної літери з'являються підказки. Вулиці відсортовані за абеткою, тож усі, що починаються з префікса, стоять **підряд** — з позиції, яку дає `bisect_left`.

**Завдання.** Допиши `suggest`: знайди початок через `bisect_left`, далі йди праворуч, поки вулиця починається з префікса (`str.startswith`) і підказок менше за `limit`.

In [ ]:
streets = sorted(["Богдана Хмельницького", "Велика Васильківська", "Верхній Вал",
                  "Володимирська", "Хрещатик", "Хорива", "Шота Руставелі"])


def suggest(streets, prefix, limit=3):
    result = []
    # YOUR CODE HERE
    # BEGIN SOLUTION
    i = bisect_left(streets, prefix)
    while i < len(streets) and streets[i].startswith(prefix) and len(result) < limit:
        result.append(streets[i])
        i += 1
    # END SOLUTION
    return result


print(suggest(streets, "В"))
assert suggest(streets, "В") == ["Велика Васильківська", "Верхній Вал", "Володимирська"]
assert suggest(streets, "Ве") == ["Велика Васильківська", "Верхній Вал"]
assert suggest(streets, "Х") == ["Хорива", "Хрещатик"]
assert suggest(streets, "Я") == []
assert suggest(streets, "В", limit=1) == ["Велика Васильківська"]
print("OK")

## 🔄 Самостійно

### Тихі години для оновлення сервера

Потрібно **k годин поспіль** з найменшою кількістю поїздок. Напиши `quietest_window(counts, k)` — ковзне вікно, що шукає мінімум. При однакових сумах повертай найраніше вікно.

In [ ]:
def quietest_window(counts, k):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    window_sum = sum(counts[0:k])
    best_start, best_sum = 0, window_sum
    for end in range(k, len(counts)):
        window_sum += counts[end] - counts[end - k]
        if window_sum < best_sum:
            best_start, best_sum = end - k + 1, window_sum
    return best_start, best_sum
    # END SOLUTION


assert quietest_window(per_hour, 2) == (2, 0)
assert quietest_window(per_hour, 4) == (1, 2)
print("OK")

### Перша поїздка з певного часу — власний `bisect_left`

Напиши бінарний пошук `first_at_or_after(items, target)` **без** модуля `bisect`: індекс першого елемента, не меншого за `target`, або `len(items)`, якщо таких немає. Порахуй порівняння: їх має бути не більше `len(items).bit_length() + 1`.

In [ ]:
def first_at_or_after(items, target):
    """Повертає (індекс, кроки)."""
    steps = 0
    # YOUR CODE HERE
    # BEGIN SOLUTION
    low, high = 0, len(items)
    while low < high:
        mid = (low + high) // 2
        steps += 1
        if items[mid] < target:
            low = mid + 1
        else:
            high = mid
    return low, steps
    # END SOLUTION


assert first_at_or_after(starts, 1080)[0] == 7
assert first_at_or_after(starts, 1400)[0] == 11
assert first_at_or_after([], 5)[0] == 0
for t in range(400, 1401):
    index, steps = first_at_or_after(starts, t)
    assert index == bisect_left(starts, t), t
    assert steps <= len(starts).bit_length() + 1
print("OK — збігається з bisect_left для всіх часів від 400 до 1400")

## ✅ Самоперевірка

1. Чому бінарний пошук не можна застосувати до невідсортованого списку?
2. Скільки кроків приблизно зробить бінарний пошук на 1 000 000 записів? А на 2 000 000?
3. Що поверне `bisect_left([10, 20, 30], 25)` і що це означає?
4. У двох вказівниках сума замала. Чому зсуваємо саме лівий вказівник?
5. Чим ковзне вікно краще за перерахунок суми кожного відрізка?

<details>
<summary>Відповіді</summary>

1. Пошук відкидає половину, спираючись на порядок. Без порядку висновок «ліва половина менша» хибний — пошук мовчки дасть неправильну відповідь.
2. Близько 20; на 2 000 000 — близько 21. Подвоєння додає один крок.
3. `2`: позиція, куди вставити 25, і водночас кількість елементів, менших за 25.
4. Правий уже на найбільшому числі. Щоб сума зросла, лишається збільшити менше — зсунути лівий праворуч.
5. Нову суму отримуємо зі старої двома діями, а не k: `O(n)` замість `O(n·k)`.

</details>

### Шпаргалка

| Що відомо про дані | Стратегія | Складність |
|---|---|---|
| нічого | лінійний пошук, `in` | `O(n)` |
| відсортовані, шукаємо значення чи позицію | бінарний пошук, `bisect_left` | `O(log n)` |
| відсортовані, шукаємо пару | два вказівники | `O(n)` |
| суцільні відрізки довжини k | ковзне вікно | `O(n)` |

## Далі

- **Урок 12 — Модулі та стандартна бібліотека.** `bisect` — лише один з багатьох готових інструментів.
- **Практикум 3 (урок 16) — Хеш-структури.** Задача про ваучер повернеться з **невідсортованими** сумами. Два вказівники вже не спрацюють, а множина знайде пару за один прохід.